In [ ]:
# ==============================================================================
# BLOQUE 1 - CARGA DE DATOS DIRECTO DESDE GITHUB
# ==============================================================================
!pip install -q transformers datasets evaluate accelerate scikit-learn

import pandas as pd
import torch

print("\n" + "="*60)
if torch.cuda.is_available():
    dispositivo = torch.cuda.get_device_name(0)
    print(f"✅ GPU detectada correctamente: {dispositivo}")
else:
    print("❌ CRÍTICO: No se detectó GPU. Ve a 'Entorno de ejecución' -> 'Cambiar tipo de entorno' y selecciona 'T4 GPU'.")
print("="*60 + "\n")

#  URL raw  de mi GitHub

url_dataset = "https://raw.githubusercontent.com/Leonardo-Rojas-Git/deteccion-estres-academico-robertuito/refs/heads/main/data_estudiantes.csv"

df = pd.read_csv(url_dataset, encoding='utf-8')
df['label'] = df['label'].astype(int)

print("✅ Dataset cargado exitosamente desde GitHub.\n")
print(f"Total de registros: {df.shape[0]}")
print(f"Total de columnas: {df.shape[1]}\n")

conteo_clases = df['label'].value_counts()
print("Distribución de la variable objetivo (label):")
print(conteo_clases)

print("\nMuestra de los primeros 3 registros:")
display(df.head(3))

In [ ]:
# ==============================================================================
# BLOQUE 2 - FASE 3: PREPARACIÓN DE DATOS (SPLIT TRIPLE Y VECTORIZACIÓN)
# ==============================================================================

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

print("\n" + "="*60)
print("INICIANDO FASE 3: SEGMENTACIÓN TRIPLE Y TOKENIZACIÓN")
print("="*60 + "\n")

# 1. División Triple del Conjunto de Datos (Train / Val / Test)
# Paso A: Separamos el 70% para Entrenamiento y el 30% restante para Temp.
df_train, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df['label']
)

# Paso B: Dividimos el 30% temporal en dos mitades iguales (15% Val / 15% Test).
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42, #parte de la estratificación
    stratify=df_temp['label'] # punto de partida para el generador de números aleatorios
)

print("Distribución del Corpus (Triple Bóveda):")
print(f"--> Entrenamiento (70%): {len(df_train)} registros (Actualiza pesos).")
print(f"--> Validación (15%): {len(df_val)} registros (Detecta Overfitting).")
print(f"--> Testeo Ciego (15%): {len(df_test)} registros (Métrica final).\n")

# 2. Conversión al formato nativo de Hugging Face (Dataset)
ds_train = Dataset.from_pandas(df_train)
ds_val = Dataset.from_pandas(df_val)
ds_test = Dataset.from_pandas(df_test)

# 3. Descarga del Tokenizador Especializado
print("Conectando con Hugging Face para descargar el tokenizador BBPE...")
nombre_modelo = "pysentimiento/robertuito-base-cased"
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)

# 4. Función de Vectorización Optimizada (max_length=96)
def tokenizar_textos(lote):
    """
    Transforma texto en vectores.
    Se reduce max_length a 96 para optimizar tiempo de cómputo y VRAM,
    siendo suficiente para la longitud promedio de comentarios digitales.
    """
    return tokenizer(
        lote["texto"],
        padding="max_length",
        truncation=True,
        max_length=96 #porque tenemos aprox. 10-50 palabras por texto.
    )                 #no pasan de 80 tokens por palabra

# 5. Ejecución del Mapeo Tensorial
print("\nVectorizando conjuntos (Train, Val, Test) a tensores de 96 posiciones...")
ds_train_tokenizado = ds_train.map(tokenizar_textos, batched=True)
ds_val_tokenizado = ds_val.map(tokenizar_textos, batched=True)
ds_test_tokenizado = ds_test.map(tokenizar_textos, batched=True)

# 6. Limpieza Estructural (Remoción de cadenas de texto)
columnas_a_borrar = ["texto", "__index_level_0__"]

# Manejo de error en caso de que '__index_level_0__' no exista
col_borrar_train = [col for col in columnas_a_borrar if col in ds_train_tokenizado.column_names]

ds_train_tokenizado = ds_train_tokenizado.remove_columns(col_borrar_train)
ds_val_tokenizado = ds_val_tokenizado.remove_columns(col_borrar_train)
ds_test_tokenizado = ds_test_tokenizado.remove_columns(col_borrar_train)

print("\n✅ ESTADO: Fase 3 completada. Datos listos para entrenamiento.")

In [ ]:
# ==============================================================================
# BLOQUE 3 - FASE 4: MODELADO Y AJUSTE FINO (FINE-TUNING)
# ==============================================================================

import numpy as np
import evaluate
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

print("\n" + "="*60)
print("INICIANDO FASE 4: CONFIGURACIÓN DE ARQUITECTURA Y ENTRENAMIENTO")
print("="*60 + "\n")

# 1. Instanciación de la Arquitectura Base
print("Descargando pesos neuronales y acoplando nueva cabeza clasificadora binaria...")
nombre_modelo = "pysentimiento/robertuito-base-cased"
modelo = AutoModelForSequenceClassification.from_pretrained(
    nombre_modelo,
    num_labels=2,
    ignore_mismatched_sizes=True
)

# 2. Definición de las Métricas de Evaluación
metrica_f1 = evaluate.load("f1")
metrica_acc = evaluate.load("accuracy")

def calcular_metricas(eval_pred):
    predicciones, etiquetas = eval_pred
    predicciones_procesadas = np.argmax(predicciones, axis=1)

    resultados_f1 = metrica_f1.compute(predictions=predicciones_procesadas, references=etiquetas)
    resultados_acc = metrica_acc.compute(predictions=predicciones_procesadas, references=etiquetas)

    return {
        "f1": resultados_f1["f1"],
        "accuracy": resultados_acc["accuracy"]
    }

# 3. Configuración de Hiperparámetros (Optimizados para Colab y Reproducibilidad)
directorio_salida = "./robertuito_estres_checkpoints"

argumentos_entrenamiento = TrainingArguments(
    output_dir=directorio_salida,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",                  # Evita cuelgues por solicitudes de API de wandb
    seed=42,                           # Garantiza resultados matemáticamente reproducibles
    save_total_limit=2,                # Evita saturar el disco del entorno temporal
)

# 4. Inicialización del Entrenador (Trainer) con Early Stopping
entrenador = Trainer(
    model=modelo,
    args=argumentos_entrenamiento,
    train_dataset=ds_train_tokenizado,
    eval_dataset=ds_val_tokenizado,
    compute_metrics=calcular_metricas,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 5. Ejecución del Entrenamiento
print("\nIniciando ajuste matemático de pesos... (Esto puede tomar entre 3 a 7 minutos)")
print("Monitoreando Validation Loss vs Training Loss para prevenir overfitting.\n")
resultados_entrenamiento = entrenador.train()

# 6. Guardado del Artefacto (local, para esta sesión de Colab)

ruta_guardado_local = "./modelo_estres_final"
entrenador.save_model(ruta_guardado_local)
tokenizer.save_pretrained(ruta_guardado_local)

print("\n" + "="*60)
print(f"✅ ESTADO: Fine-Tuning finalizado.")
print(f"Modelo guardado localmente en esta sesión de Colab en: {ruta_guardado_local}")
print("(El modelo público de referencia está en Hugging Face Hub)")
print("="*60)

In [ ]:
# ==============================================================================
# BLOQUE 4 - FASE 5: EVALUACIÓN SOBRE LA BÓVEDA CIEGA (TEST SET)
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("\n" + "="*60)
print("INICIANDO FASE 5: EVALUACIÓN FINAL SOBRE DATOS NO VISTOS")
print("="*60 + "\n")

# 1. Inferencia sobre el conjunto de testeo (15%)
print("Ejecutando inferencia neuronal sobre la bóveda de testeo...\n")
predicciones_test = entrenador.predict(ds_test_tokenizado)

# 2. Extracción de etiquetas reales vs. predicciones de la máquina
# Las predicciones vienen en decimales, usamos argmax para pasarlas a enteros (0 o 1)
etiquetas_predichas = np.argmax(predicciones_test.predictions, axis=1)
etiquetas_reales = predicciones_test.label_ids

# 3. Reporte de Clasificación Exacto
print("-" * 50)
print("REPORTE DE CLASIFICACIÓN FINAL:")
print("-" * 50)
reporte = classification_report(
    etiquetas_reales,
    etiquetas_predichas,
    target_names=["0: No Estrés", "1: Estrés"],
    digits=4 # Muestra 4 decimales para mayor precisión
)
print(reporte)

# 4. Generación de la Matriz de Confusión
print("\nGenerando Matriz de Confusión...")
matriz = confusion_matrix(etiquetas_reales, etiquetas_predichas)

plt.figure(figsize=(6, 4))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
            xticklabels=["No Estrés (0)", "Estrés (1)"],
            yticklabels=["No Estrés (0)", "Estrés (1)"],
            cbar=False, annot_kws={"size": 14})

plt.title('Matriz de Confusión - RoBERTuito (Test Set)', pad=15, fontweight='bold')
plt.ylabel('Etiqueta Real', fontweight='bold')
plt.xlabel('Predicción de la IA', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print(f"✅ FASE 5 COMPLETADA.")
print("="*60)